# 06 — Token exchange (act-on-behalf)

The agent takes alice's broad user JWT, asks Keycloak to mint a new token
that's:

- valid for **only** the target service (narrowed `aud`),
- signed by Keycloak (still cryptographic proof),
- still about alice (`sub` and `preferred_username` preserved),
- but tagged with the agent client as the requester (`azp = agent-client`),

and forwards *that* token to the service instead. RFC 8693 Token Exchange.
This is the strongest identity-propagation pattern available with standard
OAuth machinery, and as of Keycloak 26.2 it's a fully supported GA feature.

## The gap from pattern 5

Pattern 5 forwarded the user's broad JWT — proven identity, but a single
credential that covered every service the agent could call. Stolen from
one service, usable against all of them.

This pattern fixes the breadth problem. Each tool call gets its own
narrowly-scoped token, and the audit trail keeps both the user identity
(`sub`) and the agent identity (`azp`) so you can answer questions like:
*"Which agent acted on behalf of alice when this approval went through?"*

```
   user JWT (broad)            exchanged JWT (narrow)
   ----------------            ----------------------
   sub=alice                   sub=alice         <-- preserved
   aud=[expense, document]     aud=expense       <-- narrowed
   azp=agent-client            azp=agent-client  <-- audit trail
```

In [ ]:
from shared.agent import Agent
from shared.auth import TokenExchangeAuth, fetch_user_jwt, exchange_token
from shared.tools import ALL_TOOLS
from shared.display import run_as, show_what_tool_saw, show_token, compare_tokens
from shared.config import EXPENSE_SERVICE_URL, DOCUMENT_SERVICE_URL

strategy = TokenExchangeAuth()
agent = Agent(strategy=strategy, tools=ALL_TOOLS)
print(f"strategy: {strategy.name}")

## The gold-standard cell: side-by-side token comparison

Before running the agent, let's see exactly what token exchange does to
alice's JWT. The first token is what the user gets from the
identity provider (broad — usable everywhere). The second is what the
*agent* gets after exchanging the first one for the expense-service
specifically.

In [ ]:
alice_user_jwt   = fetch_user_jwt("alice")
exchanged_token  = exchange_token(alice_user_jwt, target_audience="expense-service-client")

compare_tokens(
    alice_user_jwt,
    exchanged_token,
    label_a="user JWT (broad)",
    label_b="exchanged for expense-service-client",
)

The only things that differ between these two tokens are:

- **`aud`** — narrowed from `[expense-service-client, document-service-client]`
  to just `expense-service-client`. The new token is *not* usable against
  document-service. (We'll prove this in a moment.)
- **`jti`** — different unique token IDs, as expected for two distinct tokens.

Everything else is preserved: same `sub` (alice), same `preferred_username`,
same custom claims, same `azp` identifying the agent as the requester.

This is the audit trail story for "agent acted on behalf of user".

## Proving the narrowed token is rejected by document-service

Send the exchanged (expense-service-only) token to document-service. The
document-service is configured to validate the audience, so it should
classify this token as a plain `jwt` — not a `scoped_jwt` for itself. In a
production deployment with stricter audience validation it would reject
the call entirely.

In [ ]:
import httpx
r = httpx.get(
    f"{DOCUMENT_SERVICE_URL}/documents",
    headers={"Authorization": f"Bearer {exchanged_token}"},
    timeout=5.0,
)
body = r.json()
print(f"identity_method:  {body['identity_method']}")
print(f"identity_detail:  {body['identity_detail']}")
print()
print("The token IS still cryptographically valid — Keycloak signed it — but")
print("the audience says 'this token is for expense-service-client', not for")
print("document-service-client. Real production services would reject the call")
print("at audience validation. The teaching demo just classifies it as 'jwt'")
print("(a broad token that happens to mention this service was NOT the target).")

## Same prompts, three users — full agentic flow with token exchange

In [ ]:
result_alice = run_as("alice", "What expenses do I have?", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

In [ ]:
result_bob = run_as("bob", "What expenses do my team have?", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

In [ ]:
result_carlo = run_as("carlo", "Search documents for the auth migration plan and tell me what's in it.", agent)
show_what_tool_saw(DOCUMENT_SERVICE_URL)

## Tradeoffs

- **Where authz lives:** in the tool, based on a validated *and audience-scoped*
  JWT. Same as pattern 5, but with better blast-radius properties.
- **Tool sees real user:** yes, cryptographically proven, **and** audience-bound.
  Method on the service side is `scoped_jwt`.
- **Cryptographic proof of identity:** yes.
- **Least privilege:** yes — each call carries a token usable only against
  the target service.
- **Audit trail:** the strongest available with standard OAuth. The service
  log can record `sub` (user), `azp` (the requesting agent), and `aud` (the
  intended target). You can answer *"which agent acted on behalf of alice
  to approve which expense"* directly from logs.
- **Token theft cost:** much lower than pattern 5. A token stolen from
  expense-service is useful only against expense-service.
- **Operational complexity:** higher — the realm needs the audience
  mappers configured on the requesting client, and the agent has to do a
  round-trip to Keycloak per tool call (or cache exchanged tokens with
  appropriate TTLs).

## Note on the RFC 8693 `act` claim vs Keycloak's `azp`

RFC 8693 defines an `act` claim for delegation tokens — a structured field
that says "this token represents the user, and was minted at the request
of party Y" (which can even chain: `act.act.sub` for nested delegation).

Keycloak's **Standard Token Exchange v2** (the supported, GA feature) does
not auto-populate `act` as of 26.x. The information *is* there — `azp`
identifies the requesting client and serves the same audit-trail role —
just under a different claim name. Production systems are split: some use
`azp`, some use custom claims like `x_agent_id`, some inject `act` via a
custom protocol mapper. There's no universal convention yet.

This repo deliberately uses `azp` because that's what the supported
Keycloak feature gives you natively, with no extensions. If you want the
spec-blessed `act` claim, you have two options:

1. Use Keycloak's deprecated **legacy v1** token exchange feature (which
   does populate `act`), accepting that v1 is on a deprecation path.
2. Add a custom protocol mapper that injects `act` into v2-issued tokens.

Both work; both are slightly off the supported path. The conceptual story
— *the exchanged token preserves the user identity AND identifies the
intermediate agent* — holds either way.

## What's still missing?

This pattern is genuinely strong for *what tool* a user can use. But it
can't answer questions about *which records* a user can act on:

- "Bob is a manager, so he can call `approve_expense`. Fine. But can bob
  approve **alice's** expense specifically?"
- "Alice is in engineering, so she can read engineering documents. Fine.
  But can alice read this **specific** confidential document?"

Those are *relationship* questions: the answer depends on who owns the
resource and how that owner relates to the caller. The token tells you
who's calling, but it doesn't tell you the relationship between the
caller and any specific resource.

The next notebook (`07_external_authz_tool`) flips on the `TOOL_SIDE_AUTHZ`
env var in expense-service, which makes it call OPA for fine-grained
ReBAC (Relationship-Based Access Control) decisions on each request.
That's defense-in-depth on top of token exchange.